In [ ]:
import pandas as pd
from pathlib import Path

─── Caminhos ────────────────────────────────────────────────────────────────

In [ ]:
BASE        = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
DIR_RAW     = BASE / "data/raw/ibge"
ARQUIVO_OUT = BASE / "data/processed/pnadc_2023_anual.csv"

In [ ]:
ARQUIVO_OUT.parent.mkdir(parents=True, exist_ok=True)


─── Layout oficial ───────────────────────────────────────────────────────────

In [ ]:
COLUNAS = [
    (0,    4,  "Ano",        "Ano de referência"),
    (4,    1,  "Trimestre",  "Trimestre de referência"),
    (5,    2,  "UF",         "Unidade da Federação"),
    (7,    2,  "Capital",    "Município da Capital"),
    (9,    2,  "RM_RIDE",    "Região Metropolitana / RIDE"),
    (11,   9,  "UPA",        "Unidade Primária de Amostragem"),
    (20,   7,  "Estrato",    "Estrato"),
    (27,   2,  "V1008",      "Número de seleção do domicílio"),
    (29,   2,  "V1014",      "Painel"),
    (31,   1,  "V1016",      "Número da entrevista no domicílio"),
    (49,  15,  "V1028",      "Peso COM calibração"),
    (94,   1,  "V2007",      "Sexo (1=Homem 2=Mulher)"),
    (103,  3,  "V2009",      "Idade na data de referência"),
    (106,  1,  "V2010",      "Cor ou raça"),
    (404,  1,  "VD3004",     "Nível de instrução mais elevado"),
    (408,  1,  "VD4001",     "Condição em relação à força de trabalho"),
    (409,  1,  "VD4002",     "Condição de ocupação"),
    (415,  1,  "VD4007",     "Posição na ocupação trab. principal"),
    (419,  2,  "VD4010",     "Grupamento de atividade trab. principal"),
    (135,  1,  "V4001",      "Trabalhou 1h em ativ. remunerada"),
    (150,  1,  "V4009",      "Quantos trabalhos tinha na semana"),
    (157,  5,  "V4013",      "Atividade no trabalho principal (CNAE)"),
    (194,  1,  "V4029",      "Carteira de trabalho assinada"),
    (185,  1,  "V4019",      "Negócio registrado no CNPJ"),
    (240,  3,  "V4039",      "Horas habituais no trabalho principal"),
    (243,  3,  "V4039C",     "Horas efetivas no trabalho principal"),
    (313,  3,  "V4056",      "Horas habituais no trabalho secundário"),
    (316,  3,  "V4056C",     "Horas efetivas no trabalho secundário"),
    (365,  3,  "V4062",      "Horas habituais em outros trabalhos"),
    (368,  3,  "V4062C",     "Horas efetivas em outros trabalhos"),
    (461,  3,  "VD4031",     "Horas habituais em todos os trabalhos"),
    (474,  3,  "VD4035",     "Horas efetivas em todos os trabalhos"),
    (426,  8,  "VD4016",     "Rendimento habitual trab. principal (R$)"),
    (434,  8,  "VD4017",     "Rendimento efetivo trab. principal (R$)"),
    (443,  8,  "VD4019",     "Rendimento habitual todos trabalhos (R$)"),
    (451,  8,  "VD4020",     "Rendimento efetivo todos trabalhos (R$)"),
]

In [ ]:
colspecs  = [(c[0], c[0] + c[1]) for c in COLUNAS]
col_names = [c[2] for c in COLUNAS]

In [ ]:
numericas = [
    "Ano", "Trimestre", "V2009", "V1028",
    "V4039", "V4039C", "V4056", "V4056C", "V4062", "V4062C",
    "VD4031", "VD4035",
    "VD4016", "VD4017", "VD4019", "VD4020",
]

─── Leitura e empilhamento ───────────────────────────────────────────────────

In [ ]:
arquivos = sorted(DIR_RAW.glob("PNADC_0?2023.txt"))

In [ ]:
if len(arquivos) != 4:
    print(f"ERRO: esperava 4 arquivos, encontrou {len(arquivos)}")
    print("Arquivos encontrados:", [f.name for f in arquivos])
    exit(1)

In [ ]:
trimestres = []

In [ ]:
for arq in arquivos:
    print(f"Lendo {arq.name}...", end=" ")

    df_tri = pd.read_fwf(
        arq,
        colspecs=colspecs,
        names=col_names,
        encoding="latin-1",
        dtype=str,
    )

    for col in numericas:
        df_tri[col] = pd.to_numeric(df_tri[col], errors="coerce")

    print(f"{df_tri.shape[0]:,} linhas")
    trimestres.append(df_tri)

─── Empilhar ─────────────────────────────────────────────────────────────────

In [ ]:
df = pd.concat(trimestres, ignore_index=True)
print(f"\nDataFrame anual: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

─── Validações básicas ───────────────────────────────────────────────────────

In [ ]:
print("\n─── Linhas por trimestre ─────────────────────────────")
print(df["Trimestre"].value_counts().sort_index())

In [ ]:
print("\n─── Horas habituais todos trabalhos (VD4031) ─────────")
print(df["VD4031"].describe())

In [ ]:
print("\n─── Horas habituais por trimestre (média) ────────────")
print(
    df.groupby("Trimestre")["VD4031"]
    .mean()
    .round(2)
    .to_string()
)

In [ ]:
print("\n─── Rendimento habitual todos trabalhos (VD4019) ─────")
print(df["VD4019"].describe())

In [ ]:
print("\n─── Trabalhadores ocupados (VD4002 == '1') ───────────")
ocupados = df[df["VD4002"] == "1"]
print(f"Total ocupados no ano: {len(ocupados):,}")
print(f"% do total: {len(ocupados)/len(df)*100:.1f}%")

In [ ]:
print("\n─── Horas habituais — ocupados, por trimestre ────────")
print(
    ocupados.groupby("Trimestre")["VD4031"]
    .agg(["count", "mean", "median"])
    .round(2)
    .to_string()
)

─── Distribuição de jornada — relevante para análise 6x1 ────────────────────

In [ ]:
print("\n─── Distribuição de jornada semanal (VD4031) — ocupados ─")
bins   = [0, 20, 30, 39, 40, 44, 48, 60, 120]
labels = ["até 20h", "21-30h", "31-39h", "40h", "41-44h", "45-48h", "49-60h", "60h+"]

In [ ]:
ocupados = ocupados.copy()
ocupados["faixa_horas"] = pd.cut(
    ocupados["VD4031"], bins=bins, labels=labels, right=True
)

In [ ]:
dist = (
    ocupados["faixa_horas"]
    .value_counts()
    .sort_index()
)
dist_pct = (dist / dist.sum() * 100).round(1)

In [ ]:
resumo = pd.DataFrame({"n": dist, "%": dist_pct})
print(resumo.to_string())

─── Salvar ───────────────────────────────────────────────────────────────────

In [ ]:
df.to_csv(ARQUIVO_OUT, index=False, encoding="utf-8")
print(f"\n✓ Salvo em: {ARQUIVO_OUT}")
print(f"  Tamanho: {ARQUIVO_OUT.stat().st_size / 1024 / 1024:.1f} MB")